In [2]:
import os
import time
import platform
from shutil import rmtree
from dotenv import load_dotenv

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as image
import seaborn as sns

import sklearn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, TensorDataset, random_split

import torchvision
import torchvision.transforms as transforms
import torchvision.models as models
from torchvision.datasets import ImageFolder
from torchvision.utils import make_grid

# 1. 운영체제별 한글 폰트 설정
if platform.system() == 'Windows':
    plt.rc('font', family='Malgun Gothic')
elif platform.system() == 'Darwin':  # Mac OS
    plt.rc('font', family='AppleGothic')
else:  # Linux (Colab 등)
    plt.rc('font', family='NanumBarunGothic')

# 2. 마이너스 기호(-) 깨짐 방지
plt.rcParams['axes.unicode_minus'] = False

# 오류 발생시
sklearn.set_config(display="text")

## 환경 설정

In [3]:
base_path = r"C:\kdt_0900_osg\ml\resource\dataset"
car_path = os.path.join(base_path, "f1_split")

In [4]:
load_dotenv()

print(os.environ["KAGGLE_API_TOKEN"][:7])

KGAT_99


In [13]:
import os
from kaggle.api.kaggle_api_extended import KaggleApi

api = KaggleApi()
api.authenticate()

output_dir = r"C:\kdt_0900_osg\ml\resource\dataset"
os.makedirs(output_dir, exist_ok=True)

api.dataset_download_files(
    "vesuvius13/formula-one-cars",
    path=output_dir,
    unzip=True
)

print("다운로드 완료!")

Dataset URL: https://www.kaggle.com/datasets/vesuvius13/formula-one-cars
다운로드 완료!


## 데이터 로드 & 증강 파이프라인 구성

In [5]:
# 80:20 분리
train_path = os.path.join(base_path, "f1_split", "train")
val_path = os.path.join(base_path, "f1_split", "test")

In [6]:
# 이미지 크기 224×224 및 Train 전용 증강
from torchvision import transforms

mean = [0.485, 0.456, 0.406]
std = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std)
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std)
])

In [7]:
# ImageFolder로 데이터셋 불러오기
train_dataset = ImageFolder(train_path, transform=train_transform)
val_dataset = ImageFolder(val_path, transform=val_transform)

In [8]:
print(len(train_dataset), len(val_dataset))

1096 276


In [9]:
train_dataloader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=2)
val_dataloader = DataLoader(val_dataset, batch_size=64, shuffle=False, num_workers=2)

In [10]:
# DataLoader 생성 (batch_size=32)
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False
)

In [11]:
# 클래스 수·라벨 매핑·이미지 수 및 실제 배치 확인
print("클래스 수:", len(train_dataset.classes))
print("클래스 목록:", train_dataset.classes)
print("라벨 매핑:", train_dataset.class_to_idx)

print("Train:", len(train_dataset))
print("Validation:", len(val_dataset))

images, labels = next(iter(train_loader))

print("이미지 shape:", images.shape)
print("라벨 shape:", labels.shape)

클래스 수: 4
클래스 목록: ['Ferrari F1 car', 'McLaren F1 car', 'Mercedes F1 car', 'Red Bull Racing F1 car']
라벨 매핑: {'Ferrari F1 car': 0, 'McLaren F1 car': 1, 'Mercedes F1 car': 2, 'Red Bull Racing F1 car': 3}
Train: 1096
Validation: 276
이미지 shape: torch.Size([32, 3, 224, 224])
라벨 shape: torch.Size([32])
